<a href="https://colab.research.google.com/github/christy5165/Denoising_Autoencoder.ipynb/blob/main/GEN.AI-WK-10.C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Install necessary libraries
!pip install -q transformers[torch] datasets

import torch
from datasets import Dataset
from transformers import (
    GPT2Tokenizer,
    GPT2LMHeadModel,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
    pipeline
)

# 2. Define a Technical/AI Dataset
tech_data = [
    "Machine learning is a subset of artificial intelligence focused on building systems that learn from data to improve performance over time.",
    "Deep learning utilizes neural networks with many layers to model complex patterns in large-scale datasets.",
    "Supervised learning involves training a model on a labeled dataset where the desired output is already known.",
    "Unsupervised learning looks for hidden patterns or intrinsic structures in input data without labeled responses.",
    "The primary goal of machine learning is to develop algorithms that can generalize from examples to perform tasks on unseen data."
]

# Convert to Dataset object
dataset = Dataset.from_dict({"text": tech_data})

# 3. Load GPT-2 Model and Tokenizer
model_name = "gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
model = GPT2LMHeadModel.from_pretrained(model_name)

# 4. Tokenization
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)

tokenized_datasets = dataset.map(tokenize_function, batched=True, remove_columns=["text"])
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# 5. Training Arguments
training_args = TrainingArguments(
    output_dir="./tech-gpt",
    num_train_epochs=30,          # More epochs to help it learn technical precision
    per_device_train_batch_size=2,
    save_strategy="no",
    report_to="none",
    learning_rate=3e-5
)

# 6. Fine-Tuning Process
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=tokenized_datasets,
)

print("--- Starting Technical Fine-Tuning ---")
trainer.train()
print("Fine-tuning complete!")

# 7. Generate Output for the keyword "Machine learning"
print("\n--- RESULTS: TECHNICAL GENERATION ---")
generator = pipeline('text-generation', model=model, tokenizer=tokenizer)

# Specific keyword requested in problem statement
prompt = "Machine learning"
result = generator(prompt, max_length=60, do_sample=True, temperature=0.7, truncation=True)

print(f"Keyword: {prompt}")
print("-" * 30)
print(result[0]['generated_text'])